In [1]:
%load_ext autoreload
%autoreload 2
%cd /opt/tiger/samantha

/mnt/bd/janne-research-sm/samantha


/usr/local/lib/python3.9/dist-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [6]:
!pip3 install -q dask

    PyYAML (>=5.1.*)
            ~~~~~~^


# Index

In [39]:
from samantha.utils.hdfs_helper import hdfs_ls
from tqdm import tqdm

tqdm.pandas()

In [11]:
import json
import dask.dataframe
import pandas as pd
from typing import List
from tqdm import tqdm

def read_parts(parts: List[str]):
    results = []
    for p in tqdm(parts):
        ddf = dask.dataframe.read_parquet(p)
        results.append(ddf.compute())

    results = pd.concat(results)
    results.meta = results.meta.apply(json.loads)
    return results

In [ ]:
parts = hdfs_ls("hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_Ssstk-pond5_Mnonvocal_T44k_N1608k/index_1/")

results = read_parts(parts)

In [ ]:
results["description"] = results.meta.apply(lambda i: i["description"])
results["keywords"] = results.meta.apply(lambda i: i["keywords"])
results["genres"] = results.meta.apply(lambda i: i["genres"])
results["instruments"] = results.meta.apply(lambda i: i["instruments"])
results["bpm"] = results.meta.apply(lambda i: i["bpm"])
results["quality_label"] = "high quality"

## Hard Filters

In [ ]:
MIN_DESC_LEN = 15

results = results[results.description.map(lambda r: r.isascii())]
results = results[results.description.str.len() > MIN_DESC_LEN]

results = results[~results.description.str.contains("\$")]

In [ ]:
results.keywords

## Soft Filters

In [ ]:
# preprocessing

results["bpm"] = results["bpm"].replace("\\N", 0)
results["bpm"] = results["bpm"].astype(int)

results["description"] = results["description"].replace("\\N", "")
results["keywords"] = results["keywords"].replace("\\N", "")
results["genres"] = results["genres"].replace("\\N", "")
results["instruments"] = results["instruments"].replace("\\N", "")

In [ ]:
print((results.description.str.len() == 0).sum())
print((results.keywords.str.len() == 0).sum())
print((results.genres.str.len() == 0).sum())
print((results.instruments.str.len() == 0).sum())

In [ ]:
MAX_BPM = 180
results.loc[results["bpm"] >= MAX_BPM, "quality_label"] = "bad quality"
results.loc[results["genres"].str.contains("ambient", case=False, na=False), "quality_label"] = "bad quality"
results.loc[results["keywords"].apply(lambda x: "ambient" in x), "quality_label"] = "bad quality"

In [ ]:
results.genres.value_counts()[:100].plot.barh(figsize=(20, 20))

In [ ]:
results.quality_label.value_counts().plot.barh()

## Re-write index

In [ ]:
!pip3 install -q swifter
import pandas as pd
import swifter

index_df = results.copy()

In [ ]:
index_df["meta"] = index_df.progress_apply(
    lambda row: {
        "description": row["description"],
        "keywords": row["keywords"],
        "genres": row["genres"],
        "instruments": row["instruments"],
        "bpm": row["bpm"],
        "quality_label": row["quality_label"],
    }, 
    axis=1
)

In [ ]:
index_df.meta.values[0]

In [ ]:
index_df = index_df[["uttid", "meta", "data_file", "text", "row_group_no"]]
index_df["meta"] = index_df["meta"].progress_apply(json.dumps)

In [ ]:
import os
from pathlib import Path
from tqdm import tqdm


groups = index_df.groupby("data_file")
index_version = "index_25"
for name, df in tqdm(groups, total=len(groups)):
    name = Path(name)
    
    index_name = f"{name.stem}.{index_version}"
    index_fp = f"{os.path.join(index_version, name.parent.stem, index_name)}.parquet"
    os.makedirs(os.path.dirname(index_fp), exist_ok=True)
    df.to_parquet(
        index_fp,
        index=None,
    )

In [ ]:
# hdfs dfs -put index_25 hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_Ssstk-pond5_Mnonvocal_T44k_N1608k/index_25



In [ ]:
from bytedance import easycycle

dataset_id = 7310174640181641221

easycycle.set_region(easycycle.Region.I18n)
easycycle.register_dataset_version(dataset_id, 25, 'janne.spijkervet', 'data preprocessing (v1)')

In [ ]:
from recipes.research.dataset.parquet_dataset import AudioParquetDataset
from samantha.data.audio.dataset import AudioFolderDataModule

batch_size = 8
num_workers = 16
sample_rate = 38400
shutterstock = AudioParquetDataset(
  data_id=164, # 164 = filtered, 146 = unfiltered
  sample_rate=sample_rate,
  shuffle_buffer_size=10,
  channels=2,
  pad=True,
  segment_duration=10,
  resampled=True,
  shardshuffle=True,
)
datamodule = AudioFolderDataModule([shutterstock], [shutterstock], [shutterstock], weights=None, batch_size=batch_size, shuffle=None, num_workers=num_workers)
train_loader = datamodule.train_dataloader()

In [ ]:
batch = next(iter(train_loader))
print([i["keywords"] for i in batch.index])

In [ ]:
from IPython.display import display, Audio

batch_idx = 7
n_frames = batch.segment_info[batch_idx].n_frames
display(Audio(batch.audio[batch_idx, :, :n_frames], rate=sample_rate))

## Index Dataset

In [2]:
from recipes.research.dataset.collection import ShutterStockIndexParquetDataset

dataset = ShutterStockIndexParquetDataset(
    resampled=True,
    shardshuffle=True,
)

2024-07-03 07:36:30.305726: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-07-03 07:36:30.352514: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-07-03 07:36:31.101503: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


[2024-07-03 07:36:33,202] [INFO] [real_accelerator.py:158:get_accelerator] Setting ds_accelerator to cuda (auto detect)
'FieldInfo' object has no attribute 'required'


/home/tiger/.local/lib/python3.9/site-packages/pydantic/_internal/_config.py:334: UserWarning: Valid config keys have changed in V2:
* 'allow_population_by_field_name' has been renamed to 'populate_by_name'
* 'validate_all' has been renamed to 'validate_default'
  warnings.warn(message, UserWarning)
/home/tiger/.local/lib/python3.9/site-packages/pydantic/_internal/_fields.py:160: UserWarning: Field "model_persistence_threshold" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


No module named 'monotonic_align'
2024-07-03 07:36:33,896 - bytedance.easycycle.bigspeech - INFO - region: Region.I18n
2024-07-03 07:36:33,897 - bytedance.easycycle.bigspeech - INFO - req body: {"id": 241, "arnold_trail_id": "5986203", "arnold_debug": "autonomous"}, headers: {'content-type': 'application/json'}
2024-07-03 07:36:33,955 - bytedance.easycycle.bigspeech - INFO - post https://bigspeech.byteintl.net/platform/api/v3/data/dataset_collection/get_dataset_collection_info_v2, status code: 200, body: {"origin":{"paths":[{"data":"hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_Ssstk-pond5_Mnonvocal_T44k_N1608k/data//part=*/*.parquet","index":"hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_Ssstk-pond5_Mnonvocal_T44k_N1608k/index_25//part=*/*.parquet","repetitions":"1"}]},"ratio":null,"sampling_type":"origin"}, headers: {'Date': 'Tue, 02 Jul 2024 23:36:33 GMT', 'Content-Type': 'application/json; charset=utf-8', 'Transfer-E

parse urls:   0%|          | 0/1 [00:00<?, ?it/s]

2024-07-03 07:36:35,313 - samantha.utils.hdfs_helper - INFO - Listing HDFS directory hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_Ssstk-pond5_Mnonvocal_T44k_N1608k/index_25//part=*


parse urls: 100%|██████████| 1/1 [00:02<00:00,  2.36s/it]

2024-07-03 07:36:36,411 - samantha.dataio.utils - INFO - Splitting 6291 urls by node...
2024-07-03 07:36:36,412 - samantha.dataio.utils - INFO - rank=0 world_size=1 considering node urls 6291 of total 6291
2024-07-03 07:36:36,413 - samantha.dataio.utils - INFO - rank=0 world_size=1 Preview of first 10 urls:
hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_Ssstk-pond5_Mnonvocal_T44k_N1608k/data/part=00000/shard_00001.parquet
hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_Ssstk-pond5_Mnonvocal_T44k_N1608k/data/part=00000/shard_00002.parquet
hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_Ssstk-pond5_Mnonvocal_T44k_N1608k/data/part=00000/shard_00003.parquet
hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_Ssstk-pond5_Mnonvocal_T44k_N1608k/data/part=00000/shard_00004.parquet
hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_Ssstk-pond5_Mnon

In [14]:
item = next(iter(dataset))

print()
print()
print(item.index["description"])

2024-07-03 07:45:02,860 - samantha.dataio.parquet.shardlists - INFO - resample data urls with mode self.replacement=False
2024-07-03 07:45:02,861 - samantha.dataio.parquet.shardlists - INFO - rank=0 worker=0 seed=1214902446 #0 shuffle


Disco dancing track with a retro feel of the 1970's or early 1980's. Lush and lavish, with rich string sections, funky electric guitar and occasional brass section. Reminescent of, say, early Prince, Earth Wind & Fire, or other Pop / Soul / R&B from that era.


## Full Index

In [53]:
parts = hdfs_ls("hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_Ssstk-pond5_Mnonvocal_T44k_N1608k/index_25/")
results = read_parts(parts)

2024-07-03 06:13:12,414 - samantha.utils.hdfs_helper - INFO - Listing HDFS directory hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_Ssstk-pond5_Mnonvocal_T44k_N1608k/index_25/


100%|██████████| 50/50 [00:35<00:00,  1.42it/s]


In [54]:
results["description"] = results.meta.apply(lambda i: i["description"])

In [55]:
results["description"].values[0]

'Sultry Pop track with smoky female lead vocals creating a sensual mood.'

In [22]:
results["meta"] = results.progress_apply(
    lambda row: {
        "description": row["description"],
    }, 
    axis=1
)

100%|██████████| 1515728/1515728 [00:08<00:00, 178638.54it/s]


In [25]:
results = results[["uttid", "meta", "data_file", "text", "row_group_no"]]
results["meta"] = results["meta"].progress_apply(json.dumps)

100%|██████████| 1515728/1515728 [00:05<00:00, 294245.77it/s]
/tmp/ipykernel_128412/3784995489.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results["meta"] = results["meta"].progress_apply(json.dumps)


In [26]:
results

,uttid,meta,data_file,text,row_group_no
0,158840-shutterstock,"{""description"": ""Sultry Pop track with smoky f...",../../data/part=00000/shard_00000.parquet,,0
1,141228-shutterstock,"{""description"": ""Heavy and driving, featuring ...",../../data/part=00000/shard_00000.parquet,,0
2,141229-shutterstock,"{""description"": ""Pulsing and heavy with male l...",../../data/part=00000/shard_00000.parquet,,0
3,141269-shutterstock,"{""description"": ""A long building Trap track, f...",../../data/part=00000/shard_00000.parquet,,1
4,141386-shutterstock,"{""description"": ""Smooth and jazzy with Swing e...",../../data/part=00000/shard_00000.parquet,,1
...,...,...,...,...,...
233,87799395-pond5,"{""description"": ""Instrumentation consists of t...",../../data/part=00049/shard_00104.parquet,,79
234,50001043-pond5,"{""description"": ""Cheesy, 80's-inspired new-age...",../../data/part=00049/shard_00104.parquet,,79
235,138854432-pond5,"{""description"": ""medium tempo lo-fi with sopra...",../../data/part=00049/shard_00104.parquet,,79
236,139107085-pond5,"{""description"": ""Inspiring, inspiration, inspi...",../../data/part=00049/shard_00104.parquet,,80
